In [18]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics.pairwise import cosine_similarity

Step 1: Load the Data

In [19]:
# Load the datasets
interactions = pd.read_csv('https://raw.githubusercontent.com/ywh1021/MA2_ML/refs/heads/main/interactions_train.csv')
books = pd.read_csv("https://raw.githubusercontent.com/ywh1021/MA2_ML/refs/heads/main/items.csv")
submission_sample = pd.read_csv("https://raw.githubusercontent.com/ywh1021/MA2_ML/refs/heads/main/sample_submission.csv")

# Display the first rows of each dataset
display(interactions.head())
display(books.head())

,u,i,t
0,4456,8581,1.687541e+09
1,142,1964,1.679585e+09
2,362,3705,1.706872e+09
3,1809,11317,1.673533e+09
4,4384,1323,1.681402e+09


,Title,Author,ISBN Valid,Publisher,Subjects,i
0,Classification décimale universelle : édition ...,NaN,9782871303336; 2871303339,Ed du CEFAL,Classification décimale universelle; Indexatio...,0
1,Les interactions dans l'enseignement des langu...,"Cicurel, Francine, 1947-",9782278058327; 2278058320,Didier,didactique--langue étrangère - enseignement; d...,1
2,Histoire de vie et recherche biographique : pe...,NaN,2343190194; 9782343190198,L'Harmattan,Histoires de vie en sociologie; Sciences socia...,2
3,Ce livre devrait me permettre de résoudre le c...,"Mazas, Sylvain, 1980-",9782365350020; 236535002X; 9782365350488; 2365...,Vraoum!,Moyen-Orient; Bandes dessinées autobiographiqu...,3
4,Les années glorieuses : roman /,"Lemaitre, Pierre, 1951-",9782702180815; 2702180817; 9782702183618; 2702...,Calmann-Lévy,France--1945-1975; Roman historique; Roman fra...,4


Step 2: Check the Number of interactions, users and books

In [20]:
# check number of interaction, users, books
n_users = interactions.u.nunique()
n_items = books.i.nunique()

In [21]:
print('number of users =', n_users, '| number of books =', n_items)

number of users = 7838 | number of books = 15291


Step 3: Split the Data into Training and Test Sets

In [22]:
# let's first sort the interactions by user and time stamp
interactions = interactions.sort_values(["u", "t"])
interactions["pct_rank"] = interactions.groupby("u")["t"].rank(pct=True, method='dense')
interactions.reset_index(inplace=True, drop=True)
display(interactions)

,u,i,t,pct_rank
0,0,0,1.680191e+09,0.040000
1,0,1,1.680783e+09,0.080000
2,0,2,1.680801e+09,0.120000
3,0,3,1.683715e+09,0.160000
4,0,3,1.683715e+09,0.200000
...,...,...,...,...
87042,7836,3471,1.728644e+09,0.666667
87043,7836,3471,1.728644e+09,1.000000
87044,7837,2191,1.728735e+09,0.333333
87045,7837,88,1.728735e+09,0.666667


In [23]:
# Define a function to create the data matrix
def create_data_matrix(data, n_users, n_items):
    data_matrix = np.zeros((n_users, n_items))
    data_matrix[data["u"].values, data["i"].values] = 1

    return data_matrix

In [24]:
# Define the function to predict interactions based on item similarity
def item_based_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions.T) / (similarity.sum(axis=1)[:, np.newaxis] + epsilon) # epsilon is for avoiding dividing 0
    return pred.T  # Transpose to get users as rows and items as columns

In [25]:
# Define the function to predict interactions based on user similarity
def user_based_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions) / (np.abs(similarity).sum(axis=1)[:, np.newaxis] + epsilon)
    return pred

In [26]:
# Implement the precision_recall_at_k function
def precision_recall_at_k(prediction, ground_truth, k=10):
    num_users = prediction.shape[0]
    precision_at_k, recall_at_k = 0, 0

    for user in range(num_users):
        top_k_items = np.argpartition(prediction[user], -k)[-k:]
        relevant_items_in_top_k = np.sum(ground_truth[user][top_k_items])
        total_relevant_items = np.sum(ground_truth[user])

        # Update Precision@K and Recall@K for this user
        precision_at_k += relevant_items_in_top_k / k
        recall_at_k += relevant_items_in_top_k / total_relevant_items if total_relevant_items > 0 else 0

    # Calculate the average Precision@K and Recall@K over all users
    precision_at_k /= num_users
    recall_at_k /= num_users

    return precision_at_k, recall_at_k

In [27]:
# ==========================================
# 5-Fold Cross Validation (Time-based per user)
# ==========================================
import gc

k_folds = 5
precision_user_cv = []
recall_user_cv = []
precision_item_cv = []
recall_item_cv = []

print(f"\nStarting {k_folds}-Fold Cross Validation...")

for fold in range(k_folds):
    # set fold percentage border
    test_lower = fold / k_folds
    test_upper = (fold + 1) / k_folds

    # use pct_rank separate train and test set (separate user time to cut data)
    # make sure to cover 1.0，the last fold using <=
    if fold == k_folds - 1:
        test_mask = (interactions["pct_rank"] >= test_lower) & (interactions["pct_rank"] <= test_upper)
    else:
        test_mask = (interactions["pct_rank"] >= test_lower) & (interactions["pct_rank"] < test_upper)

    train_mask = ~test_mask

    train_data_cv = interactions[train_mask]
    test_data_cv = interactions[test_mask]
    print()

    # build Data Matrices
    train_matrix_cv = create_data_matrix(train_data_cv, n_users, n_items)
    test_matrix_cv = create_data_matrix(test_data_cv, n_users, n_items)

    # -------------------------------------------
    # Item-Based CF
    # -------------------------------------------
    item_sim_cv = cosine_similarity(train_matrix_cv.T)
    item_pred_cv = item_based_predict(train_matrix_cv, item_sim_cv)
    prec_item, rec_item = precision_recall_at_k(item_pred_cv, test_matrix_cv, k=10)

    precision_item_cv.append(prec_item)
    recall_item_cv.append(rec_item)

    del item_sim_cv, item_pred_cv
    gc.collect()

    # -------------------------------------------
    # User-Based CF
    # -------------------------------------------
    user_sim_cv = cosine_similarity(train_matrix_cv)
    user_pred_cv = user_based_predict(train_matrix_cv, user_sim_cv)
    prec_user, rec_user = precision_recall_at_k(user_pred_cv, test_matrix_cv, k=10)

    precision_user_cv.append(prec_user)
    recall_user_cv.append(rec_user)

    del user_sim_cv, user_pred_cv
    gc.collect()

    print(f"Fold {fold + 1} completed | "
          f"Fold {fold + 1} # of train dataset: {len(train_data_cv)}, # of test dataset : {len(test_data_cv)}"
          f"Test range: [{test_lower:.1f}, {test_upper:.1f}) | "
          f"Item P@10: {prec_item:.4f}, User P@10: {prec_user:.4f}")

# ==========================================
# Result output
# ==========================================
print("\n=== 5-Fold Cross Validation Results ===")
print(f"Average User-based CF Precision@10: {np.mean(precision_user_cv):.4f} ± {np.std(precision_user_cv):.4f}")
print(f"Average User-based CF Recall@10:    {np.mean(recall_user_cv):.4f} ± {np.std(recall_user_cv):.4f}")
print("-" * 40)
print(f"Average Item-based CF Precision@10: {np.mean(precision_item_cv):.4f} ± {np.std(precision_item_cv):.4f}")
print(f"Average Item-based CF Recall@10:    {np.mean(recall_item_cv):.4f} ± {np.std(recall_item_cv):.4f}")


Starting 5-Fold Cross Validation...

Fold 1 completed | Fold 1 # of train dataset: 74423, # of test dataset : 12624Test range: [0.0, 0.2) | Item P@10: 0.0323, User P@10: 0.0297

Fold 2 completed | Fold 2 # of train dataset: 68779, # of test dataset : 18268Test range: [0.2, 0.4) | Item P@10: 0.0506, User P@10: 0.0514

Fold 3 completed | Fold 3 # of train dataset: 70789, # of test dataset : 16258Test range: [0.4, 0.6) | Item P@10: 0.0458, User P@10: 0.0458

Fold 4 completed | Fold 4 # of train dataset: 68778, # of test dataset : 18269Test range: [0.6, 0.8) | Item P@10: 0.0539, User P@10: 0.0550

Fold 5 completed | Fold 5 # of train dataset: 65419, # of test dataset : 21628Test range: [0.8, 1.0) | Item P@10: 0.0557, User P@10: 0.0566

=== 5-Fold Cross Validation Results ===
Average User-based CF Precision@10: 0.0477 ± 0.0097
Average User-based CF Recall@10:    0.2616 ± 0.0675
----------------------------------------
Average Item-based CF Precision@10: 0.0477 ± 0.0084
Average Item-based C

In [28]:
# filter top 10 prediction
def get_top_10_df(prediction, k=10):
    num_users = prediction.shape[0]
    recommendation_list = []

    for user in range(num_users):
        # Step 1: get top 10 index
        top_k_items = np.argsort(prediction[user])[-k:][::-1]

        # Step 2: use " " to separate strings
        rec_str = " ".join(top_k_items.astype(str))

        # Step 3: save user_id and recommendation strings
        recommendation_list.append({
            "user_id": user,
            "recommendation": rec_str
        })

    # Step 4: transform to dataframe
    df = pd.DataFrame(recommendation_list)
    return df

## User-to-User Collaborative Filtering (All interaction data used)

In [29]:
# Step 1: Compute User Similarity Matrix
train_data_matrix_all = create_data_matrix(interactions, n_users, n_items)
# Compute the user-user similarity matrix
user_similarity_all = cosine_similarity(train_data_matrix_all)

# Calculate the user-based predictions for positive interactions
user_prediction_all = user_based_predict(train_data_matrix_all, user_similarity_all)

In [30]:
top_10_user_df_all = get_top_10_df(user_prediction_all, k=10)

# 2. preview the results
print(top_10_user_df_all.head())

# 3. csv
# top_10_user_df_all.to_csv("recommendations_user_alldata.csv", index=False)

   user_id                           recommendation
0        0                13 4 12 23 15 11 14 8 5 9
1        1            38 39 30 31 34 36 37 32 33 29
2        2            46 58 49 56 53 91 64 87 45 71
3        3   149 169 163 167 128 133 143 40 139 165
4        4  203 198 207 205 195 202 193 191 199 201
